# 01: Pressure-to-Deformation Calibration Model

## Overview
This notebook establishes the mathematical relationship between applied pneumatic pressure (mbar) and microfluidic wall deformation. This calibration curve allows us to treat the PDMS chamber as a physical force sensor.

### Technical Approach:
- Metrology: Uses scipy.spatial.distance.cdist for robust pairwise distance measurement between non-parallel wall boundaries.
- Output: Generates a linear/polynomial calibration model saved as a CSV for biological analysis.

In [ ]:
import os
import numpy as np
import pandas as pd
import csv
from skimage.io import imread
from skimage.measure import label, regionprops
from scipy.spatial.distance import cdist
from natsort import natsorted

# 1. Setup Generic Paths
base_dir = os.path.dirname(os.getcwd())
input_folder = os.path.join(base_dir, "data", "raw", "calibration")
output_csv = os.path.join(base_dir, "data", "results", "Calibration_Model.csv")

os.makedirs(os.path.dirname(output_csv), exist_ok=True)
pixel_size = 0.06587  # µm/pixel

def calculate_wall_distance(mask_image):
    """Calculates average Euclidean distance between two segmented walls."""
    labeled = label(mask_image > 0)
    props = regionprops(labeled)
    if len(props) < 2: return None, None
    
    # Get coordinates of the two largest objects (the walls)
    wall1_coords = props[0].coords
    wall2_coords = props[1].coords
    
    # Compute pairwise distances
    distances = cdist(wall1_coords, wall2_coords, 'euclidean')
    min_distances = np.min(distances, axis=1)
    return np.mean(min_distances) * pixel_size, np.std(min_distances) * pixel_size

# 2. Processing Loop
results = []
if os.path.exists(input_folder):
    # Natural sorting handles 'Calibration-10' correctly after 'Calibration-2'
    files = natsorted([f for f in os.listdir(input_folder) if f.endswith('.png')])
    # Pressure mapping based on experimental setup (0 to 2000 mbar)
    pressures = [0, 200, 400, 600, 800, 1000, 1500, 2000]
    
    for i, file in enumerate(files):
        img = imread(os.path.join(input_folder, file))
        dist, std = calculate_wall_distance(img)
        if dist:
            results.append({"Pressure_mbar": pressures[i], "Distance_um": dist, "Std_um": std})

pd.DataFrame(results).to_csv(output_csv, index=False)
print(f"Calibration model saved to {output_csv}")